# Census EDA

Quick look at `data/census.csv` before training. Goal: predict `salary` (`<=50K` / `>50K`).

In [1]:
import pandas as pd

raw = pd.read_csv('data/census.csv')
raw.columns.tolist()

['age',
 ' workclass',
 ' fnlgt',
 ' education',
 ' education-num',
 ' marital-status',
 ' occupation',
 ' relationship',
 ' race',
 ' sex',
 ' capital-gain',
 ' capital-loss',
 ' hours-per-week',
 ' native-country',
 ' salary']

Every column name (and every value) has a leading space, e.g. `' salary'`. That's what the README means by messy.

`skipinitialspace=True` fixes it on load, so no need to edit the csv by hand.

In [2]:
df = pd.read_csv('data/census.csv', skipinitialspace=True)
print(df.shape)
df.head()

(32561, 15)


,age,workclass,fnlgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
df.dtypes

age                int64
workclass         object
fnlgt              int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
salary            object
dtype: object

## Missing values

No NaNs, but `?` is used as the unknown marker.

In [4]:
df.isna().sum().sum()

np.int64(0)

In [5]:
(df == '?').sum()[lambda s: s > 0]

workclass         1836
occupation        1843
native-country     583
dtype: int64

Keeping `?` as its own category. A real user might not know someone's occupation either, and the one-hot encoder handles it fine.

## Duplicates

In [6]:
df.duplicated().sum()

np.int64(24)

24 out of 32k, not worth worrying about.

## Target

In [7]:
df['salary'].value_counts(normalize=True)

salary
<=50K    0.75919
>50K     0.24081
Name: proportion, dtype: float64

~76 / 24. Always guessing `<=50K` already gives 76% accuracy, so I'll look at precision / recall / F1 instead.

## Salary by group

Share of `>50K` per category. Big gaps here are worth checking later in the slice metrics.

In [8]:
high = df['salary'].eq('>50K')
high.groupby(df['education']).mean().sort_values(ascending=False)

education
Doctorate       0.740920
Prof-school     0.734375
Masters         0.556587
Bachelors       0.414753
Assoc-voc       0.261216
Assoc-acdm      0.248360
Some-college    0.190235
HS-grad         0.159509
12th            0.076212
10th            0.066452
7th-8th         0.061920
9th             0.052529
11th            0.051064
5th-6th         0.048048
1st-4th         0.035714
Preschool       0.000000
Name: salary, dtype: float64

In [9]:
high.groupby(df['sex']).mean()

sex
Female    0.109461
Male      0.305737
Name: salary, dtype: float64

In [10]:
high.groupby(df['race']).mean().sort_values(ascending=False)

race
Asian-Pac-Islander    0.265640
White                 0.255860
Black                 0.123880
Amer-Indian-Eskimo    0.115756
Other                 0.092251
Name: salary, dtype: float64

## Full profile

Saved to `census_report.html` (gitignored, rerun this to regenerate).

In [11]:
from data_profiling import ProfileReport

profile = ProfileReport(df, title='Census Report')
profile.to_file('census_report.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

100%|██████████| 15/15 [00:00<00:00, 3266.42it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]